In [2]:
# Define working directory to make pathing easier later

import os
os.chdir('../data')

In [13]:
# Import, standardize labels, and combine data for SNOTEL and SNODAS

#Combine swe data
import pandas as pd
import glob
import os

# Load SNODAS
snodas_df = pd.read_csv(r'SNODAS/SNODAS_swe_all.csv')
snodas_df['date'] = pd.to_datetime(snodas_df['date'])

# Load SNOTEL
snotel_df = pd.read_csv('SNOTEL/SNOTEL_swe_20042024_cleaned.csv', comment='#', parse_dates=['date'])
snotel_df = snotel_df[['date', 'swe_m']].rename(columns={'swe_m': 'swe_m_snotel'})
snotel_df['date'] = pd.to_datetime(snotel_df['date'])
print(snotel_df.head())

# Merge on date
merged_df = pd.merge(snodas_df, snotel_df, on='date', how='inner')
merged_df = merged_df.sort_values('date').reset_index(drop=True)

print(f'{len(merged_df)} matched records')
print(merged_df.head())

# Save
merged_df.to_csv(r'MERGED/swe_merged.csv', index=False)


        date  swe_m_snotel
0 2004-03-01       0.34036
1 2004-03-02       0.34544
2 2004-03-03       0.35306
3 2004-03-04       0.35306
4 2004-03-05       0.36322
2552 matched records
        date  swe_m_snodas  swe_m_snotel
0 2004-03-01         0.325       0.34036
1 2004-03-02         0.353       0.34544
2 2004-03-03         0.353       0.35306
3 2004-03-04         0.353       0.35306
4 2004-03-05         0.369       0.36322


In [8]:
#Summary Stats

print('=== SHAPE ===')
print(f'Rows: {len(merged_df)}, Expected: ~2562')

print('\n=== MISSING VALUES ===')
print(merged_df.isnull().sum())

print('\n=== DUPLICATE DATES ===')
dupes = merged_df[merged_df.duplicated(subset='date', keep=False)]
print(f'Duplicate date rows: {len(dupes)}')
if len(dupes) > 0:
    print(dupes)

print('\n=== DESCRIPTIVE STATISTICS ===')
print(merged_df[['swe_m_snodas', 'swe_m_snotel']].describe())

print('\n=== OUTLIERS (values > 5m or < 0) ===')
outliers = merged_df[(merged_df['swe_m_snodas'] > 5) | (merged_df['swe_m_snodas'] < 0) |
                     (merged_df['swe_m_snotel'] > 5) | (merged_df['swe_m_snotel'] < 0)]
print(f'Outlier rows: {len(outliers)}')
if len(outliers) > 0:
    print(outliers)

print('\n=== CORRELATION (SNODAS vs SNOTEL) ===')
corr = merged_df['swe_m_snodas'].corr(merged_df['swe_m_snotel'])
print(f'Pearson correlation: {corr:.4f}')

print('\n=== DIFFERENCE STATISTICS (SNODAS - SNOTEL) ===')
merged_df['diff'] = merged_df['swe_m_snodas'] - merged_df['swe_m_snotel']
print(merged_df['diff'].describe())

=== SHAPE ===
Rows: 2552, Expected: ~2562

=== MISSING VALUES ===
date            0
swe_m_snodas    0
swe_m_snotel    0
diff            0
dtype: int64

=== DUPLICATE DATES ===
Duplicate date rows: 0

=== DESCRIPTIVE STATISTICS ===
       swe_m_snodas  swe_m_snotel
count   2552.000000   2552.000000
mean       0.483517      0.484610
std        0.209494      0.216926
min        0.000000      0.000000
25%        0.363000      0.378460
50%        0.501000      0.510540
75%        0.628000      0.619760
max        1.135000      0.942340

=== OUTLIERS (values > 5m or < 0) ===
Outlier rows: 0

=== CORRELATION (SNODAS vs SNOTEL) ===
Pearson correlation: 0.8882

=== DIFFERENCE STATISTICS (SNODAS - SNOTEL) ===
count    2552.000000
mean       -0.001093
std         0.101056
min        -0.248340
25%        -0.063255
50%        -0.006720
75%         0.056030
max         0.382740
Name: diff, dtype: float64


In [10]:
# Data understanding/cleaning
# missing dates?
import pandas as pd
from datetime import datetime


# Generate all expected dates (Mar 1 - Jun 30 for each year)
expected_dates = []
for year in range(2004, 2025):
    year_dates = pd.date_range(datetime(year, 3, 1), datetime(year, 6, 30), freq='1D')
    expected_dates.extend(year_dates)

expected_df = pd.DataFrame({'date': expected_dates})

# Find missing dates
missing = expected_df[~expected_df['date'].isin(merged_df['date'])]
print(f'{len(missing)} missing dates')

# Check which source is missing for each date
for date in missing['date']:
    in_snodas = date in snodas_df['date'].values
    in_snotel = date in snotel_df['date'].values
    print(f'{date.date()} — SNODAS: {in_snodas}, SNOTEL: {in_snotel}')

10 missing dates
2005-06-25 — SNODAS: False, SNOTEL: True
2005-06-26 — SNODAS: False, SNOTEL: True
2005-06-27 — SNODAS: False, SNOTEL: True
2005-06-28 — SNODAS: False, SNOTEL: True
2005-06-29 — SNODAS: False, SNOTEL: True
2005-06-30 — SNODAS: False, SNOTEL: True
2007-03-26 — SNODAS: False, SNOTEL: True
2008-03-13 — SNODAS: False, SNOTEL: True
2008-06-13 — SNODAS: False, SNOTEL: True
2008-06-18 — SNODAS: False, SNOTEL: True


In [15]:
# merge missing with swe_merged.csv
recovered_df = pd.read_csv(r'SNODAS/SNODAS_swe_missingdates.csv', parse_dates=['date'])
print(recovered_df)

        date  swe_m
0 2005-06-26    0.0
1 2005-06-27    0.0
2 2005-06-28    0.0
3 2005-06-29    0.0
4 2005-06-30    0.0


In [17]:
# Merge recovered dates with SNOTEL
recovered_merged = pd.merge(recovered_df, snotel_df, on='date', how='inner')
recovered_merged = recovered_merged.rename(columns={'swe_m': 'swe_m_snodas'})

# Load existing merged CSV
merged_df = pd.read_csv(r'MERGED/swe_merged.csv', parse_dates=['date'])

# Append, deduplicate, sort
merged_df = pd.concat([merged_df, recovered_merged]).drop_duplicates(subset='date').sort_values('date').reset_index(drop=True)

print(f'{len(merged_df)} total records')

# Save
merged_df.to_csv(r'MERGED/swe_merged2.csv', index=False)

2557 total records


In [5]:
# day of year column
import pandas as pd
from datetime import datetime

peaks_df = pd.read_csv(r'MERGED/peaks_swe.csv', parse_dates=['peak_date_snodas', 'peak_date_snotel'])

peaks_df['peak_doy_snodas'] = peaks_df['peak_date_snodas'].dt.dayofyear
peaks_df['peak_doy_snotel'] = peaks_df['peak_date_snotel'].dt.dayofyear
peaks_df['peak_doy_diff'] = peaks_df['peak_doy_snodas'] - peaks_df['peak_doy_snotel']

print(peaks_df[['year', 'peak_date_snodas', 'peak_doy_snodas', 
                 'peak_date_snotel', 'peak_doy_snotel', 
                 'peak_doy_diff']].to_string(index=False))
# Save
peaks_df.to_csv(r'MERGED/peaks_swe.csv', index=False)

 year peak_date_snodas  peak_doy_snodas peak_date_snotel  peak_doy_snotel  peak_doy_diff
 2004       2004-04-25              116       2004-04-23              114              2
 2005       2005-05-04              124       2005-05-13              133             -9
 2006       2006-04-19              109       2006-04-19              109              0
 2007       2007-05-02              122       2007-04-27              117              5
 2008       2008-06-14              166       2008-05-25              146             20
 2009       2009-05-16              136       2009-05-15              135              1
 2010       2010-05-14              134       2010-05-13              133              1
 2011       2011-06-01              152       2011-05-31              151              1
 2012       2012-05-08              129       2012-05-06              127              2
 2013       2013-05-21              141       2013-04-24              114             27
 2014       2014-05-0